In [13]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
 

In [15]:

df = pd.read_csv('Customers_Data.csv')
df.head()

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   CustomerID              200 non-null    int64
 1   Gender                  200 non-null    str  
 2   Age                     200 non-null    int64
 3   Annual Income (k$)      200 non-null    int64
 4   Spending Score (1-100)  200 non-null    int64
dtypes: int64(4), str(1)
memory usage: 7.9 KB


In [17]:
df.isnull().sum()

CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64

In [18]:
df_clean = df.drop_duplicates().copy()
df_clean = df_clean.dropna()
# Standardize column names
df_clean.columns = [c.strip() for c in df_clean.columns]
print("\nShape after cleaning: ", df_clean.shape)


Shape after cleaning:  (200, 5)


In [20]:
numeric_df = df_clean.select_dtypes(include=[np.number]).drop(columns=["CustomerID"], errors="ignore")
corr = numeric_df.corr()
print("\nCorrelation matrix")
print(corr)
 



Correlation matrix
                             Age  Annual Income (k$)  Spending Score (1-100)
Age                     1.000000           -0.012398               -0.327227
Annual Income (k$)     -0.012398            1.000000                0.009903
Spending Score (1-100) -0.327227            0.009903                1.000000


In [35]:
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
OUT = "."
plt.savefig(f"{OUT}/01_correlation_heatmap.png", dpi=150)
plt.close()

In [36]:
sns.pairplot(df_clean, vars=["Age","Annual Income (k$)","Spending Score (1-100)"], hue="Gender", diag_kind="kde")
plt.savefig(f"{OUT}/02_pairplot.png", dpi=150)
plt.close()

In [37]:
selected_features = ["Annual Income (k$)", "Spending Score (1-100)"]
print("\nSelected features for clustering (based on low mutual correlation & business relevance):", selected_features)
 
X = df_clean[selected_features].values


Selected features for clustering (based on low mutual correlation & business relevance): ['Annual Income (k$)', 'Spending Score (1-100)']


In [38]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 

In [ ]:


inertias = []
sil_scores = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
 
fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
axes[0].plot(list(K_range), inertias, marker='o')
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow Method")
 
axes[1].plot(list(K_range), sil_scores, marker='o', color='darkorange')
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Analysis")
plt.tight_layout()
plt.savefig(f"{OUT}/03_optimal_k.png", dpi=150)
plt.close()
 
best_k = list(K_range)[int(np.argmax(sil_scores))]
print("\nInertias:", dict(zip(K_range, inertias)))
print("Silhouette scores:", dict(zip(K_range, sil_scores)))
print("Best k by silhouette:", best_k)

#  Measures distance inside a single cluster.Checks if dots crowd the center closely.Lower scores mean tighter customer groups.
# Measures distance between neighboring clusters.Checks if different groups overlap too much.Higher scores mean cleaner group boundaries.Reveals a single, clear peak value.
# Inertia stops you from making too few clusters.Silhouette stops you from making too many clusters.Together, they find the perfect middle ground.


Inertias: {2: 269.69101219276394, 3: 157.70400815035947, 4: 108.92131661364357, 5: 65.5684081557168, 6: 55.05734827038599, 7: 44.86475569922556, 8: 37.228187677585886, 9: 32.39226763033116, 10: 29.981897788243693}
Silhouette scores: {2: 0.3212707813918878, 3: 0.46658474419000145, 4: 0.4939069237513199, 5: 0.5546571631111091, 6: 0.5398800926790663, 7: 0.5281492781108291, 8: 0.4552147906587443, 9: 0.4570853966942764, 10: 0.4431713026508046}
Best k by silhouette: 5


In [29]:

final_k = best_k
kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
df_clean["Cluster"] = cluster_labels
 
print(f"\nFinal model: KMeans with k={final_k}")
print("Cluster sizes:\n", df_clean["Cluster"].value_counts().sort_index())


Final model: KMeans with k=5
Cluster sizes:
 Cluster
0    81
1    39
2    22
3    35
4    23
Name: count, dtype: int64


In [40]:

plt.figure(figsize=(7,6))
palette = sns.color_palette("Set2", final_k)
for c in range(final_k):
    subset = df_clean[df_clean["Cluster"] == c]
    plt.scatter(subset["Annual Income (k$)"], subset["Spending Score (1-100)"],
                s=60, color=palette[c], label=f"Cluster {c}")
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers_original[:,0], centers_original[:,1], s=250, c='black', marker='X', label='Centroids')
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title(f"Customer Segments (KMeans, k={final_k})")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/04_clusters_scatter.png", dpi=150)
plt.close()

In [32]:

plt.figure(figsize=(7,4.5))
sns.boxplot(data=df_clean, x="Cluster", y="Age", palette="Set2")
plt.title("Age Distribution by Cluster")
plt.tight_layout()
plt.savefig(f"{OUT}/05_age_by_cluster.png", dpi=150)
plt.close()

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_8188\3507176504.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_clean, x="Cluster", y="Age", palette="Set2")


In [41]:
summary = df_clean.groupby("Cluster")[["Age","Annual Income (k$)","Spending Score (1-100)"]].mean().round(1)
summary["Count"] = df_clean["Cluster"].value_counts().sort_index()
print("\n--- Cluster Profile Summary ---")
print(summary)
 
summary.to_csv(f"{OUT}/cluster_summary.csv")
df_clean.to_csv(f"{OUT}/Customers_Data_Clustered.csv", index=False)
 



--- Cluster Profile Summary ---
          Age  Annual Income (k$)  Spending Score (1-100)  Count
Cluster                                                         
0        42.7                55.3                    49.5     81
1        32.7                86.5                    82.1     39
2        25.3                25.7                    79.4     22
3        41.1                88.2                    17.1     35
4        45.2                26.3                    20.9     23
